# E-Commerce Analytics Project
## 03 — Exploratory Business Analysis

### Objective
Analyze the cleaned e-commerce data to understand business performance, identify major revenue and profitability drivers, evaluate customer and product behavior, and uncover insights that can support management decisions.

### Core Business Questions
- How are revenue and profit changing over time?
- Which products and categories drive the most revenue and profit?
- Which categories are hurt most by returns and discounts?
- How do new and returning customers perform differently?
- Which acquisition channels generate the most valuable customers?
- Which regions contribute most to company performance?
- Where are the strongest opportunities to improve profitability?

In [1]:
import pandas as pd
import numpy as np
import sqlite3

from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATABASE_PATH = PROJECT_ROOT / "database" / "ecommerce.db"

In [3]:
customers = pd.read_csv(
    PROCESSED_DIR / "customers_clean.csv",
    parse_dates=["signup_date"]
)

products = pd.read_csv(
    PROCESSED_DIR / "products_clean.csv"
)

orders = pd.read_csv(
    PROCESSED_DIR / "orders_clean.csv",
    parse_dates=["order_date"]
)

order_items = pd.read_csv(
    PROCESSED_DIR / "order_items_clean.csv"
)

returns = pd.read_csv(
    PROCESSED_DIR / "returns_clean.csv",
    parse_dates=["return_date"]
)

marketing = pd.read_csv(
    PROCESSED_DIR / "marketing_clean.csv",
    parse_dates=["date"]
)

In [4]:
transactions = (
    order_items
    .merge(
        orders,
        on="order_id",
        how="left"
    )
    .merge(
        products,
        on="product_id",
        how="left"
    )
    .merge(
        customers,
        on="customer_id",
        how="left"
    )
)

In [5]:
transactions.shape

(125273, 24)

In [6]:
transactions.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,customer_id,order_date,device,sales_channel,...,category,subcategory,unit_cost,list_price,signup_date,city,state,region,age_group,acquisition_channel
0,ITEM-0000001,ORD-000001,PROD-0124,1,225.92,0.30,CUST-20202,2025-03-05,Desktop,Website,...,Fashion,Sneakers,149.15,225.92,2023-05-12,Port Gabriel,TX,South,25-34,Affiliate
1,ITEM-0000002,ORD-000001,PROD-0045,1,233.26,0.05,CUST-20202,2025-03-05,Desktop,Website,...,Home & Kitchen,Cookware Set,126.19,233.26,2023-05-12,Port Gabriel,TX,South,25-34,Affiliate
2,ITEM-0000003,ORD-000001,PROD-0058,1,31.18,0.00,CUST-20202,2025-03-05,Desktop,Website,...,Sports & Outdoors,Yoga Mat,22.88,31.18,2023-05-12,Port Gabriel,TX,South,25-34,Affiliate
3,ITEM-0000004,ORD-000002,PROD-0012,1,302.76,0.30,CUST-18773,2024-12-05,Desktop,Website,...,Electronics,Webcam,130.02,302.76,2024-10-28,Davidfurt,PA,Northeast,45-54,Social Media
4,ITEM-0000005,ORD-000002,PROD-0115,2,47.22,0.00,CUST-18773,2024-12-05,Desktop,Website,...,Fashion,Sneakers,26.66,47.22,2024-10-28,Davidfurt,PA,Northeast,45-54,Social Media


In [7]:
transactions["gross_sales"] = (
    transactions["quantity"]
    * transactions["unit_price"]
)

transactions["discount_amount"] = (
    transactions["gross_sales"]
    * transactions["discount_pct"]
)

transactions["revenue"] = (
    transactions["gross_sales"]
    - transactions["discount_amount"]
)

transactions["cogs"] = (
    transactions["quantity"]
    * transactions["unit_cost"]
)

transactions["gross_profit"] = (
    transactions["revenue"]
    - transactions["cogs"]
)

transactions["gross_margin_pct"] = np.where(
    transactions["revenue"] != 0,
    transactions["gross_profit"]
    / transactions["revenue"] * 100,
    np.nan
)

In [8]:
completed_transactions = transactions[
    transactions["order_status"] == "Completed"
].copy()

In [9]:
total_revenue = completed_transactions["revenue"].sum()

gross_profit = completed_transactions["gross_profit"].sum()

orders_count = completed_transactions["order_id"].nunique()

customers_count = completed_transactions["customer_id"].nunique()

units_sold = completed_transactions["quantity"].sum()

aov = total_revenue / orders_count

gross_margin = (
    gross_profit / total_revenue * 100
)

In [10]:
print(f"Revenue: ${total_revenue:,.2f}")
print(f"Gross Profit: ${gross_profit:,.2f}")
print(f"Gross Margin: {gross_margin:.2f}%")
print(f"Orders: {orders_count:,}")
print(f"Customers: {customers_count:,}")
print(f"Units Sold: {units_sold:,}")
print(f"Average Order Value: ${aov:,.2f}")

Revenue: $26,495,886.38
Gross Profit: $11,293,659.33
Gross Margin: 42.62%
Orders: 72,756
Customers: 19,521
Units Sold: 168,301
Average Order Value: $364.17


## 1. Revenue and Growth Analysis

Business performance will first be evaluated over time to determine:

- Revenue growth by year
- Monthly revenue trends
- Year-over-year growth
- Month-over-month growth
- Seasonality
- Gross profit trends
- Gross margin trends

In [11]:
completed_transactions["year"] = (
    completed_transactions["order_date"].dt.year
)

completed_transactions["month"] = (
    completed_transactions["order_date"].dt.to_period("M")
)

In [12]:
yearly_performance = (
    completed_transactions
    .groupby("year")
    .agg(
        revenue=("revenue", "sum"),
        gross_profit=("gross_profit", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique"),
        units_sold=("quantity", "sum")
    )
)

yearly_performance["gross_margin_pct"] = (
    yearly_performance["gross_profit"]
    / yearly_performance["revenue"]
    * 100
)

yearly_performance["revenue_growth_pct"] = (
    yearly_performance["revenue"]
    .pct_change()
    * 100
)

yearly_performance.round(2)

,revenue,gross_profit,orders,customers,units_sold,gross_margin_pct,revenue_growth_pct
year,,,,,,,
2023,7497268.62,3197645.60,20497,6320,47484,42.65,NaN
2024,8788323.82,3738984.98,24188,11792,55899,42.54,17.22
2025,10210293.94,4357028.74,28071,16149,64918,42.67,16.18


In [13]:
monthly_performance = (
    completed_transactions
    .groupby("month")
    .agg(
        revenue=("revenue", "sum"),
        gross_profit=("gross_profit", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
)

monthly_performance["month"] = (
    monthly_performance["month"]
    .dt.to_timestamp()
)

monthly_performance["mom_growth_pct"] = (
    monthly_performance["revenue"]
    .pct_change()
    * 100
)

monthly_performance.head(12)

,month,revenue,gross_profit,orders,mom_growth_pct
0,2023-01-01,467865.8870,196758.2070,1301,NaN
1,2023-02-01,414288.8720,176553.8420,1143,-11.451362
2,2023-03-01,567475.8415,242238.8315,1540,36.975883
3,2023-04-01,583882.9145,252032.9545,1533,2.891237
4,2023-05-01,585248.4575,247637.0075,1619,0.233873
5,2023-06-01,596095.1650,253901.5950,1657,1.853351
6,2023-07-01,647599.6720,276105.9420,1808,8.640316
7,2023-08-01,618996.8665,262451.0065,1698,-4.416742
8,2023-09-01,587518.1935,247968.1435,1631,-5.085433
9,2023-10-01,621521.4865,264853.3865,1681,5.787615


In [14]:
monthly_performance["revenue_3m_avg"] = (
    monthly_performance["revenue"]
    .rolling(3)
    .mean()
)

In [15]:
yearly_performance["aov"] = (
    yearly_performance["revenue"]
    / yearly_performance["orders"]
)

yearly_performance["units_per_order"] = (
    yearly_performance["units_sold"]
    / yearly_performance["orders"]
)

yearly_performance[
    [
        "revenue",
        "orders",
        "aov",
        "units_per_order",
        "gross_margin_pct",
        "revenue_growth_pct"
    ]
].round(2)

,revenue,orders,aov,units_per_order,gross_margin_pct,revenue_growth_pct
year,,,,,,
2023,7497268.62,20497,365.77,2.32,42.65,NaN
2024,8788323.82,24188,363.33,2.31,42.54,17.22
2025,10210293.94,28071,363.73,2.31,42.67,16.18


In [16]:
completed_transactions["month_number"] = (
    completed_transactions["order_date"].dt.month
)

monthly_seasonality = (
    completed_transactions
    .groupby(
        ["year", "month_number"]
    )
    .agg(
        revenue=("revenue", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
)

monthly_seasonality.head()

,year,month_number,revenue,orders
0,2023,1,467865.8870,1301
1,2023,2,414288.8720,1143
2,2023,3,567475.8415,1540
3,2023,4,583882.9145,1533
4,2023,5,585248.4575,1619


In [17]:
seasonality_pivot = (
    monthly_seasonality
    .pivot(
        index="month_number",
        columns="year",
        values="revenue"
    )
)

seasonality_pivot.round(2)

year,2023,2024,2025
month_number,,,
1,467865.89,531977.18,656838.29
2,414288.87,561620.80,605530.16
3,567475.84,676821.32,715620.99
4,583882.91,675007.01,783397.95
5,585248.46,667344.33,830187.12
6,596095.16,704425.95,827591.09
7,647599.67,782961.63,891696.14
8,618996.87,686490.03,832056.02
9,587518.19,685712.19,841236.21


In [18]:
monthly_performance["yoy_growth_pct"] = (
    monthly_performance["revenue"]
    .pct_change(12)
    * 100
)

In [19]:
monthly_performance[
    [
        "month",
        "revenue",
        "mom_growth_pct",
        "yoy_growth_pct",
        "revenue_3m_avg"
    ]
].round(2)

,month,revenue,mom_growth_pct,yoy_growth_pct,revenue_3m_avg
0,2023-01-01,467865.89,NaN,NaN,NaN
1,2023-02-01,414288.87,-11.45,NaN,NaN
2,2023-03-01,567475.84,36.98,NaN,483210.20
3,2023-04-01,583882.91,2.89,NaN,521882.54
4,2023-05-01,585248.46,0.23,NaN,578869.07
5,2023-06-01,596095.16,1.85,NaN,588408.85
6,2023-07-01,647599.67,8.64,NaN,609647.76
7,2023-08-01,618996.87,-4.42,NaN,620897.23
8,2023-09-01,587518.19,-5.09,NaN,618038.24
9,2023-10-01,621521.49,5.79,NaN,609345.52


## 2. Product & Category Performance

This section evaluates which product categories and individual products drive company revenue, volume, and gross profit.

Key questions:
- Which categories generate the most revenue?
- Which categories generate the most gross profit?
- Which categories have the strongest margins?
- Which products are the top revenue drivers?
- Are high-revenue products also the most profitable?

In [20]:
category_performance = (
    completed_transactions
    .groupby("category")
    .agg(
        revenue=("revenue", "sum"),
        gross_profit=("gross_profit", "sum"),
        units_sold=("quantity", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique"),
        discount_amount=("discount_amount", "sum")
    )
)

In [21]:
category_performance["gross_margin_pct"] = (
    category_performance["gross_profit"]
    / category_performance["revenue"]
    * 100
)

category_performance["avg_discount_pct"] = (
    category_performance["discount_amount"]
    / (
        category_performance["revenue"]
        + category_performance["discount_amount"]
    )
    * 100
)

category_performance = (
    category_performance
    .round(2)
    .sort_values(
        "revenue",
        ascending=False
    )
)

category_performance

,revenue,gross_profit,units_sold,orders,customers,discount_amount,gross_margin_pct,avg_discount_pct
category,,,,,,,,
Electronics,6846336.21,3035091.14,39337,25187,13190,686183.10,44.33,9.11
Home & Kitchen,6175535.52,2648837.76,39322,25060,13239,600340.93,42.89,8.86
Beauty & Health,5886739.53,2495887.62,33242,21481,12182,584157.59,42.40,9.03
Sports & Outdoors,3844103.68,1594172.16,24250,16227,10326,384177.11,41.47,9.09
Fashion,3743171.43,1519670.63,32150,20997,12000,364918.74,40.60,8.88


In [22]:
category_performance["revenue_share_pct"] = (
    category_performance["revenue"]
    / category_performance["revenue"].sum()
    * 100
)

category_performance.round(2)

,revenue,gross_profit,units_sold,orders,customers,discount_amount,gross_margin_pct,avg_discount_pct,revenue_share_pct
category,,,,,,,,,
Electronics,6846336.21,3035091.14,39337,25187,13190,686183.10,44.33,9.11,25.84
Home & Kitchen,6175535.52,2648837.76,39322,25060,13239,600340.93,42.89,8.86,23.31
Beauty & Health,5886739.53,2495887.62,33242,21481,12182,584157.59,42.40,9.03,22.22
Sports & Outdoors,3844103.68,1594172.16,24250,16227,10326,384177.11,41.47,9.09,14.51
Fashion,3743171.43,1519670.63,32150,20997,12000,364918.74,40.60,8.88,14.13


In [23]:
product_performance = (
    completed_transactions
    .groupby(
        [
            "product_id",
            "product_name",
            "category"
        ]
    )
    .agg(
        revenue=("revenue", "sum"),
        gross_profit=("gross_profit", "sum"),
        units_sold=("quantity", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
)

product_performance["gross_margin_pct"] = (
    product_performance["gross_profit"]
    / product_performance["revenue"]
    * 100
)

In [24]:
top_products = (
    product_performance
    .sort_values(
        "revenue",
        ascending=False
    )
    .head(10)
)

top_products.round(2)

,product_id,product_name,category,revenue,gross_profit,units_sold,orders,gross_margin_pct
49,PROD-0050,Webcam 50,Electronics,538231.73,285296.90,1629,1179,53.01
113,PROD-0114,Webcam 114,Electronics,531809.60,272794.88,1632,1178,51.30
103,PROD-0104,Headphones 104,Electronics,487911.33,226145.67,2078,1474,46.35
42,PROD-0043,Smart Watch 43,Electronics,487650.24,248175.20,1616,1184,50.89
11,PROD-0012,Webcam 12,Electronics,470019.76,246385.36,1720,1241,52.42
78,PROD-0079,Webcam 79,Electronics,449726.14,236867.27,1312,948,52.67
32,PROD-0033,Sneakers 33,Fashion,425700.71,221501.32,1733,1274,52.03
52,PROD-0053,Desk Lamp 53,Home & Kitchen,408048.06,197050.35,1377,986,48.29
41,PROD-0042,Bluetooth Speaker 42,Electronics,390087.75,185022.75,1550,1134,47.43
108,PROD-0109,Hair Dryer 109,Beauty & Health,379703.52,204854.07,1357,979,53.95


In [25]:
top_profit_products = (
    product_performance
    .sort_values(
        "gross_profit",
        ascending=False
    )
    .head(10)
)

top_profit_products.round(2)

,product_id,product_name,category,revenue,gross_profit,units_sold,orders,gross_margin_pct
49,PROD-0050,Webcam 50,Electronics,538231.73,285296.90,1629,1179,53.01
113,PROD-0114,Webcam 114,Electronics,531809.60,272794.88,1632,1178,51.30
42,PROD-0043,Smart Watch 43,Electronics,487650.24,248175.20,1616,1184,50.89
11,PROD-0012,Webcam 12,Electronics,470019.76,246385.36,1720,1241,52.42
78,PROD-0079,Webcam 79,Electronics,449726.14,236867.27,1312,948,52.67
103,PROD-0104,Headphones 104,Electronics,487911.33,226145.67,2078,1474,46.35
32,PROD-0033,Sneakers 33,Fashion,425700.71,221501.32,1733,1274,52.03
108,PROD-0109,Hair Dryer 109,Beauty & Health,379703.52,204854.07,1357,979,53.95
52,PROD-0053,Desk Lamp 53,Home & Kitchen,408048.06,197050.35,1377,986,48.29
136,PROD-0137,Water Bottle 137,Sports & Outdoors,363682.21,190029.83,1166,832,52.25


## 3. Returns & Refund Impact

This section evaluates how product returns affect sales performance.

Key questions:
- Which categories have the highest return rates?
- How much revenue is refunded?
- Which categories lose the most sales value to returns?
- Do category rankings change after accounting for refunds?

In [26]:
refunds_by_item = (
    returns
    .groupby("order_item_id")
    .agg(
        refund_amount=("refund_amount", "sum")
    )
    .reset_index()
)

In [27]:
transactions_returns = (
    completed_transactions
    .merge(
        refunds_by_item,
        on="order_item_id",
        how="left"
    )
)

transactions_returns["refund_amount"] = (
    transactions_returns["refund_amount"]
    .fillna(0)
)

In [28]:
transactions_returns["was_returned"] = (
    transactions_returns["refund_amount"] > 0
)

In [29]:
transactions_returns["net_revenue_after_refunds"] = (
    transactions_returns["revenue"]
    - transactions_returns["refund_amount"]
)

In [30]:
category_return_impact = (
    transactions_returns
    .groupby("category")
    .agg(
        gross_revenue=("revenue", "sum"),
        refunds=("refund_amount", "sum"),
        net_revenue=("net_revenue_after_refunds", "sum"),
        items=("order_item_id", "count"),
        returned_items=("was_returned", "sum")
    )
)

In [31]:
category_return_impact["return_rate_pct"] = (
    category_return_impact["returned_items"]
    / category_return_impact["items"]
    * 100
)

category_return_impact["refund_pct_of_revenue"] = (
    category_return_impact["refunds"]
    / category_return_impact["gross_revenue"]
    * 100
)

category_return_impact.round(2).sort_values(
    "refund_pct_of_revenue",
    ascending=False
)

,gross_revenue,refunds,net_revenue,items,returned_items,return_rate_pct,refund_pct_of_revenue
category,,,,,,,
Fashion,3743171.43,542210.98,3200960.45,23225,3418,14.72,14.49
Electronics,6846336.21,548449.37,6297886.84,28570,2283,7.99,8.01
Sports & Outdoors,3844103.68,298040.67,3546063.01,17540,1394,7.95,7.75
Home & Kitchen,6175535.52,437941.63,5737593.89,28301,1973,6.97,7.09
Beauty & Health,5886739.53,296954.44,5589785.09,23889,1201,5.03,5.04


## 4. Customer Performance Analysis

This section evaluates customer purchasing behavior and value.

Key questions:
- How many customers are new versus returning?
- How much revenue do returning customers generate?
- Do returning customers have higher average order values?
- Which acquisition channels produce the most valuable customers?
- How concentrated is revenue among high-value customers?

In [32]:
customer_orders = (
    completed_transactions
    .groupby("customer_id")
    .agg(
        first_order=("order_date", "min"),
        last_order=("order_date", "max"),
        orders=("order_id", "nunique"),
        revenue=("revenue", "sum"),
        units=("quantity", "sum")
    )
    .reset_index()
)

In [33]:
customer_orders["customer_type"] = np.where(
    customer_orders["orders"] > 1,
    "Returning",
    "One-Time"
)

In [34]:
customer_type_summary = (
    customer_orders
    .groupby("customer_type")
    .agg(
        customers=("customer_id", "nunique"),
        total_orders=("orders", "sum"),
        revenue=("revenue", "sum")
    )
)

In [35]:
customer_type_summary["revenue_per_customer"] = (
    customer_type_summary["revenue"]
    / customer_type_summary["customers"]
)

In [36]:
customer_type_summary["revenue_share_pct"] = (
    customer_type_summary["revenue"]
    / customer_type_summary["revenue"].sum()
    * 100
)

customer_type_summary.round(2)

,customers,total_orders,revenue,revenue_per_customer,revenue_share_pct
customer_type,,,,,
One-Time,4721,4721,1747054.72,370.06,6.59
Returning,14800,68035,24748831.67,1672.22,93.41


In [37]:
customer_type_lookup = customer_orders[
    ["customer_id", "customer_type"]
]

customer_transactions = (
    completed_transactions
    .merge(
        customer_type_lookup,
        on="customer_id",
        how="left"
    )
)

In [38]:
customer_type_performance = (
    customer_transactions
    .groupby("customer_type")
    .agg(
        revenue=("revenue", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique")
    )
)

customer_type_performance["aov"] = (
    customer_type_performance["revenue"]
    / customer_type_performance["orders"]
)

customer_type_performance["orders_per_customer"] = (
    customer_type_performance["orders"]
    / customer_type_performance["customers"]
)

customer_type_performance.round(2)

,revenue,orders,customers,aov,orders_per_customer
customer_type,,,,,
One-Time,1747054.72,4721,4721,370.06,1.0
Returning,24748831.67,68035,14800,363.77,4.6


## 5. Customer Acquisition Channel Analysis

This section evaluates the quality and value of customers acquired through different channels.

Key questions:
- Which channels acquire the most customers?
- Which channels generate the most revenue?
- Which channels produce the highest-value customers?
- Which channels generate the strongest repeat purchasing behavior?

In [39]:
channel_performance = (
    customer_transactions
    .groupby("acquisition_channel")
    .agg(
        revenue=("revenue", "sum"),
        gross_profit=("gross_profit", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique")
    )
)

In [40]:
channel_performance["revenue_per_customer"] = (
    channel_performance["revenue"]
    / channel_performance["customers"]
)

channel_performance["aov"] = (
    channel_performance["revenue"]
    / channel_performance["orders"]
)

channel_performance["orders_per_customer"] = (
    channel_performance["orders"]
    / channel_performance["customers"]
)

channel_performance["gross_margin_pct"] = (
    channel_performance["gross_profit"]
    / channel_performance["revenue"]
    * 100
)

channel_performance.round(2).sort_values(
    "revenue",
    ascending=False
)

,revenue,gross_profit,orders,customers,revenue_per_customer,aov,orders_per_customer,gross_margin_pct
acquisition_channel,,,,,,,,
Email,4689542.89,1990865.52,12763,3306,1418.49,367.43,3.86,42.45
Social Media,4423305.83,1881214.02,12206,3246,1362.69,362.39,3.76,42.53
Paid Search,4400949.96,1880965.84,12034,3278,1342.57,365.71,3.67,42.74
Organic Search,4357773.31,1860372.19,11981,3268,1333.47,363.72,3.67,42.69
Affiliate,4326818.09,1850279.79,11873,3170,1364.93,364.43,3.75,42.76
Direct,4297496.31,1829961.97,11899,3253,1321.09,361.16,3.66,42.58


In [41]:
customer_channel = (
    customer_orders
    .merge(
        customers[
            ["customer_id", "acquisition_channel"]
        ],
        on="customer_id",
        how="left"
    )
)

channel_retention = (
    customer_channel
    .groupby("acquisition_channel")
    .agg(
        customers=("customer_id", "nunique"),
        returning_customers=(
            "customer_type",
            lambda x: (x == "Returning").sum()
        )
    )
)

channel_retention["returning_customer_pct"] = (
    channel_retention["returning_customers"]
    / channel_retention["customers"]
    * 100
)

channel_retention.round(2).sort_values(
    "returning_customer_pct",
    ascending=False
)

,customers,returning_customers,returning_customer_pct
acquisition_channel,,,
Social Media,3246,2491,76.74
Email,3306,2533,76.62
Affiliate,3170,2402,75.77
Paid Search,3278,2482,75.72
Direct,3253,2445,75.16
Organic Search,3268,2447,74.88


## 6. Marketing Efficiency Analysis

Marketing performance will be compared with attributed customer acquisition and revenue.

Key metrics include:
- Marketing Spend
- Customers Acquired
- Customer Acquisition Cost (CAC)
- Attributed Revenue
- Return on Ad Spend (ROAS)

### Attribution Limitation
Direct customers do not have associated campaign spend, while Display Ads spend does not have directly attributed customers in the customer dataset.

CAC and attributed ROAS will therefore be calculated only for channels with matching customer and marketing attribution.

In [42]:
marketing_spend = (
    marketing
    .groupby("channel")
    .agg(
        marketing_spend=("spend", "sum"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum")
    )
)

marketing_spend["ctr_pct"] = (
    marketing_spend["clicks"]
    / marketing_spend["impressions"]
    * 100
)

marketing_spend.round(2).sort_values(
    "marketing_spend",
    ascending=False
)

,marketing_spend,impressions,clicks,ctr_pct
channel,,,,
Paid Search,1793952.23,104497142,4404131.0,4.21
Social Media,1331907.28,102292284,3063286.0,2.99
Display Ads,766022.80,106047388,1309823.0,1.24
Affiliate,547603.89,104658603,3615338.0,3.45
Email,187637.20,105691366,5793335.0,5.48
Organic Search,0.00,102001564,4514586.0,4.43


In [43]:
customers_acquired = (
    customers
    .groupby("acquisition_channel")
    .agg(
        customers_acquired=("customer_id", "nunique")
    )
)

customers_acquired

,customers_acquired
acquisition_channel,
Affiliate,4076
Direct,4172
Email,4246
Organic Search,4184
Paid Search,4163
Social Media,4159


In [44]:
marketing_efficiency = (
    marketing_spend
    .reset_index()
    .rename(
        columns={"channel": "acquisition_channel"}
    )
    .merge(
        customers_acquired.reset_index(),
        on="acquisition_channel",
        how="outer"
    )
)

In [45]:
channel_revenue = (
    channel_performance[
        ["revenue", "gross_profit"]
    ]
    .reset_index()
)

marketing_efficiency = (
    marketing_efficiency
    .merge(
        channel_revenue,
        on="acquisition_channel",
        how="outer"
    )
)

In [46]:
marketing_efficiency.round(2)

,acquisition_channel,marketing_spend,impressions,clicks,ctr_pct,customers_acquired,revenue,gross_profit
0,Affiliate,547603.89,104658603.0,3615338.0,3.45,4076.0,4326818.09,1850279.79
1,Direct,NaN,NaN,NaN,NaN,4172.0,4297496.31,1829961.97
2,Display Ads,766022.80,106047388.0,1309823.0,1.24,NaN,NaN,NaN
3,Email,187637.20,105691366.0,5793335.0,5.48,4246.0,4689542.89,1990865.52
4,Organic Search,0.00,102001564.0,4514586.0,4.43,4184.0,4357773.31,1860372.19
5,Paid Search,1793952.23,104497142.0,4404131.0,4.21,4163.0,4400949.96,1880965.84
6,Social Media,1331907.28,102292284.0,3063286.0,2.99,4159.0,4423305.83,1881214.02


In [47]:
marketing_efficiency["cac"] = np.where(
    (
        marketing_efficiency["marketing_spend"].notna()
        &
        marketing_efficiency["customers_acquired"].notna()
        &
        (marketing_efficiency["customers_acquired"] > 0)
    ),
    marketing_efficiency["marketing_spend"]
    / marketing_efficiency["customers_acquired"],
    np.nan
)

In [48]:
marketing_efficiency["roas"] = np.where(
    (
        marketing_efficiency["marketing_spend"].notna()
        &
        (marketing_efficiency["marketing_spend"] > 0)
        &
        marketing_efficiency["revenue"].notna()
    ),
    marketing_efficiency["revenue"]
    / marketing_efficiency["marketing_spend"],
    np.nan
)

In [49]:
marketing_efficiency[
    [
        "acquisition_channel",
        "marketing_spend",
        "customers_acquired",
        "cac",
        "revenue",
        "roas",
        "ctr_pct"
    ]
].round(2).sort_values(
    "roas",
    ascending=False
)

,acquisition_channel,marketing_spend,customers_acquired,cac,revenue,roas,ctr_pct
3,Email,187637.20,4246.0,44.19,4689542.89,24.99,5.48
0,Affiliate,547603.89,4076.0,134.35,4326818.09,7.90,3.45
6,Social Media,1331907.28,4159.0,320.25,4423305.83,3.32,2.99
5,Paid Search,1793952.23,4163.0,430.93,4400949.96,2.45,4.21
1,Direct,NaN,4172.0,NaN,4297496.31,NaN,NaN
2,Display Ads,766022.80,NaN,NaN,NaN,NaN,1.24
4,Organic Search,0.00,4184.0,0.00,4357773.31,NaN,4.43


## 7. Geographic Performance Analysis

This section evaluates business performance across U.S. regions and states.

Key questions:
- Which regions generate the most revenue?
- Which states have the strongest sales performance?
- Which regions have the highest average order values?
- Are certain regions more profitable than others?
- Where are the strongest opportunities for geographic expansion?

In [50]:
region_performance = (
    completed_transactions
    .groupby("region")
    .agg(
        revenue=("revenue", "sum"),
        gross_profit=("gross_profit", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique"),
        units_sold=("quantity", "sum")
    )
)

In [51]:
region_performance["aov"] = (
    region_performance["revenue"]
    / region_performance["orders"]
)

region_performance["revenue_per_customer"] = (
    region_performance["revenue"]
    / region_performance["customers"]
)

region_performance["gross_margin_pct"] = (
    region_performance["gross_profit"]
    / region_performance["revenue"]
    * 100
)

region_performance["revenue_share_pct"] = (
    region_performance["revenue"]
    / region_performance["revenue"].sum()
    * 100
)

region_performance.round(2).sort_values(
    "revenue",
    ascending=False
)

,revenue,gross_profit,orders,customers,units_sold,aov,revenue_per_customer,gross_margin_pct,revenue_share_pct
region,,,,,,,,,
West,7979375.50,3394513.50,21973,5843,50617,363.14,1365.63,42.54,30.12
Midwest,6658741.09,2838788.61,18263,4909,42335,364.60,1356.44,42.63,25.13
South,6508344.14,2783257.32,17875,4832,41326,364.10,1346.93,42.76,24.56
Northeast,5349425.64,2277099.88,14645,3937,34023,365.27,1358.76,42.57,20.19


In [52]:
state_performance = (
    completed_transactions
    .groupby("state")
    .agg(
        revenue=("revenue", "sum"),
        gross_profit=("gross_profit", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique")
    )
)

state_performance["aov"] = (
    state_performance["revenue"]
    / state_performance["orders"]
)

state_performance["gross_margin_pct"] = (
    state_performance["gross_profit"]
    / state_performance["revenue"]
    * 100
)

state_performance.round(2).sort_values(
    "revenue",
    ascending=False
).head(10)

,revenue,gross_profit,orders,customers,aov,gross_margin_pct
state,,,,,,
MN,1444246.66,620502.66,3899,1037,370.41,42.96
AZ,1425382.48,610009.02,3928,1039,362.88,42.80
NJ,1405172.56,601182.66,3906,1041,359.75,42.78
IL,1362743.59,579609.01,3712,996,367.12,42.53
PA,1358769.79,576557.10,3643,977,372.98,42.43
OR,1358072.54,576496.94,3766,961,360.61,42.45
MI,1355035.87,577594.52,3754,1014,360.96,42.63
CO,1353565.79,581831.45,3727,982,363.18,42.99
MA,1338468.62,568692.56,3699,1001,361.85,42.49


## 8. RFM Customer Segmentation

RFM analysis evaluates customers based on:

- Recency: How recently did the customer purchase?
- Frequency: How often does the customer purchase?
- Monetary Value: How much revenue has the customer generated?

The goal is to identify high-value customers, loyal customers, new customers, and customers potentially at risk of churn.

In [53]:
analysis_date = (
    completed_transactions["order_date"].max()
    + pd.Timedelta(days=1)
)

analysis_date

Timestamp('2026-01-01 00:00:00')

In [54]:
rfm = (
    completed_transactions
    .groupby("customer_id")
    .agg(
        last_purchase=("order_date", "max"),
        frequency=("order_id", "nunique"),
        monetary=("revenue", "sum")
    )
    .reset_index()
)

In [55]:
rfm["recency"] = (
    analysis_date
    - rfm["last_purchase"]
).dt.days

In [56]:
rfm[
    [
        "customer_id",
        "recency",
        "frequency",
        "monetary"
    ]
].head()

,customer_id,recency,frequency,monetary
0,CUST-00001,173,2,657.5900
1,CUST-00002,375,3,1832.6165
2,CUST-00003,27,6,2123.6130
3,CUST-00004,1,7,2303.0610
4,CUST-00007,96,7,1911.8635


In [57]:
rfm["r_score"] = pd.qcut(
    rfm["recency"],
    5,
    labels=[5, 4, 3, 2, 1]
)

rfm["f_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm["m_score"] = pd.qcut(
    rfm["monetary"],
    5,
    labels=[1, 2, 3, 4, 5]
)

In [58]:
rfm[
    ["r_score", "f_score", "m_score"]
] = rfm[
    ["r_score", "f_score", "m_score"]
].astype(int)

In [59]:
rfm["rfm_score"] = (
    rfm["r_score"]
    + rfm["f_score"]
    + rfm["m_score"]
)

In [63]:
def assign_segment(row):

    if (
        row["r_score"] >= 4
        and row["f_score"] >= 4
        and row["m_score"] >= 4
    ):
        return "Champions"

    elif (
        row["f_score"] >= 4
        and row["m_score"] >= 4
    ):
        return "High-Value Lapsed"

    elif (
        row["r_score"] >= 4
        and row["frequency"] <= 2
    ):
        return "New / Promising"

    elif (
        row["r_score"] <= 2
        and row["f_score"] >= 3
    ):
        return "At Risk"

    elif (
        row["r_score"] <= 2
        and row["f_score"] <= 2
    ):
        return "Low Engagement"

    else:
        return "Regular"


rfm["segment"] = rfm.apply(
    assign_segment,
    axis=1
)

In [64]:
rfm_segments = (
    rfm
    .groupby("segment")
    .agg(
        customers=("customer_id", "count"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_customer_value=("monetary", "mean"),
        total_revenue=("monetary", "sum")
    )
)

In [65]:
rfm_segments["revenue_share_pct"] = (
    rfm_segments["total_revenue"]
    / rfm_segments["total_revenue"].sum()
    * 100
)

rfm_segments.round(2).sort_values(
    "total_revenue",
    ascending=False
)

,customers,avg_recency,avg_frequency,avg_customer_value,total_revenue,revenue_share_pct
segment,,,,,,
High-Value Lapsed,3613,287.98,6.85,2645.54,9558322.91,36.07
Champions,2689,42.49,7.03,2697.20,7252764.74,27.37
Regular,4518,100.94,2.75,878.29,3968112.38,14.98
At Risk,2224,365.17,3.36,1065.34,2369322.05,8.94
New / Promising,3289,43.99,1.45,533.19,1753654.76,6.62
Low Engagement,3188,373.02,1.39,499.91,1593709.55,6.01


## 9. Executive Insights & Recommendations

The analysis identified several major business trends, risks, and opportunities.

The following insights summarize the most important findings for management and will guide the final executive dashboard.

### Key Insights

1. **Revenue growth remains strong and consistent.**  
   Revenue increased 17.2% in 2024 and another 16.2% in 2025 while gross margin remained stable at approximately 42.6%.

2. **Growth is primarily driven by transaction volume rather than higher order values.**  
   Average Order Value remained near $364 across the three-year period.

3. **The business has strong seasonal dependence on the holiday period.**  
   November and December consistently generate substantially higher revenue than typical months.

4. **Fashion has the largest returns problem.**  
   Approximately 14.5% of Fashion revenue is refunded, nearly twice the overall company return impact.

5. **Customer retention is a major revenue driver.**  
   Returning customers generate 93.4% of total revenue and approximately 4.5x more revenue per customer than one-time buyers.

6. **Email is the most efficient attributable marketing channel.**  
   Email generates approximately $24.99 in attributed customer revenue for every $1 of marketing spend, with a CAC of only $44.19.

7. **High-value customers represent a major reactivation opportunity.**  
   High-Value Lapsed customers account for 36.1% of historical revenue but average approximately 288 days since their last purchase.

8. **Geographic differences are driven mostly by customer volume.**  
   The West generates the most revenue, but AOV and gross margins remain highly consistent across regions.

### Management Recommendations

1. **Prioritize customer retention and reactivation.**  
   Develop targeted campaigns for High-Value Lapsed and At-Risk customers, since historically valuable customers represent a significant share of revenue.

2. **Investigate the root causes of Fashion returns.**  
   Review sizing accuracy, product descriptions, imagery, product quality, and return reasons to reduce the category's unusually high refund rate.

3. **Increase emphasis on high-efficiency marketing channels.**  
   Evaluate opportunities to expand Email and Affiliate marketing while reviewing the comparatively high acquisition cost of Paid Search.

4. **Improve attribution for Display Ads.**  
   Approximately $766K of Display Ads spending cannot currently be directly linked to acquired customers or revenue, limiting management's ability to evaluate channel effectiveness.

5. **Plan inventory and marketing around holiday seasonality.**  
   November and December consistently produce the strongest sales periods, making inventory planning and campaign timing especially important.

6. **Protect and grow Champion customers.**  
   Use loyalty benefits, personalized recommendations, and cross-sell strategies to maintain engagement among the company's highest-value active customers.

In [66]:
ANALYSIS_DIR = PROJECT_ROOT / "data" / "analysis"

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [67]:
yearly_performance.to_csv(
    ANALYSIS_DIR / "yearly_performance.csv"
)

monthly_performance.to_csv(
    ANALYSIS_DIR / "monthly_performance.csv",
    index=False
)

category_performance.to_csv(
    ANALYSIS_DIR / "category_performance.csv"
)

product_performance.to_csv(
    ANALYSIS_DIR / "product_performance.csv",
    index=False
)

category_return_impact.to_csv(
    ANALYSIS_DIR / "category_return_impact.csv"
)

customer_type_performance.to_csv(
    ANALYSIS_DIR / "customer_type_performance.csv"
)

marketing_efficiency.to_csv(
    ANALYSIS_DIR / "marketing_efficiency.csv",
    index=False
)

region_performance.to_csv(
    ANALYSIS_DIR / "region_performance.csv"
)

state_performance.to_csv(
    ANALYSIS_DIR / "state_performance.csv"
)

rfm.to_csv(
    ANALYSIS_DIR / "rfm_customers.csv",
    index=False
)

rfm_segments.to_csv(
    ANALYSIS_DIR / "rfm_segments.csv"
)

print("Analysis tables exported successfully.")

Analysis tables exported successfully.
